# LLM Fine-Tuning on AMD MI300X (192GB VRAM)

This notebook provides a complete pipeline for fine-tuning a massive LLM on your Java Vulnerability Dataset.
Because you have an incredible 192GB of VRAM, we have upgraded the default model to **Qwen2.5-Coder-32B-Instruct** and drastically increased the batch sizes to utilize your compute power.

In [ ]:
!pip install -U transformers peft trl datasets accelerate wandb
!pip install bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import wandb

In [ ]:
# 1. Load the Dataset
data_files = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl"
}
dataset = load_dataset("json", data_files=data_files)
print(dataset)

In [ ]:
# 2. Load Tokenizer & Apply Chat Template
model_id = "Qwen/Qwen2.5-Coder-32B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    if "messages" in example:
        example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    else:
        instruction = example.get('prompt', '') or example.get('instruction', '')
        input_code = example.get('input', '')
        completion = example.get('completion', '') or example.get('output', '')
        
        # Combine instruction and input code into the user prompt
        full_prompt = f"{instruction}\n\n{input_code}" if input_code else instruction
        
        # CRITICAL FIX: Explicitly Label Safe Code!
        # If the expected output is identical to the input, it means the code is safe.
        # Instead of silently returning the same code, we teach the model to explicitly declare it is safe.
        if input_code and input_code.strip() == completion.strip():
            completion = "This Java code is completely secure and contains no vulnerabilities. No changes are required."
            
        messages = [
            {"role": "user", "content": full_prompt},
            {"role": "assistant", "content": completion}
        ]
        example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

formatted_dataset = dataset.map(format_chat_template)

In [ ]:
# 3. Load Model with 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.config.use_cache = False 
model = prepare_model_for_kbit_training(model)

In [ ]:
# 4. Setup LoRA Configuration
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [ ]:
# 5. Configure Training Arguments using SFTConfig
training_args = SFTConfig(
    output_dir="./large-java-vuln-model",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=500,                       
    num_train_epochs=1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",                    
    dataset_text_field="text"            
)

In [ ]:
# 6. Initialize Trainer and Start Training
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['validation'],
    peft_config=peft_config,             
    processing_class=tokenizer,
    args=training_args,
)

trainer.model.print_trainable_parameters() 
print("Starting heavy training loop on MI300X...")
trainer.train()

In [ ]:
# 7. Save the Fine-Tuned Adapter Weights
trainer.model.save_pretrained("large-java-vuln-adapter")
tokenizer.save_pretrained("large-java-vuln-adapter")
print("Training Complete! Adapter saved to './large-java-vuln-adapter'")